---
## 순환 그래프 — AI끼리 대화 시켜보기

15.2의 **A ↔ B ↔ judge** 순환 구조에 LangChain LLM을 넣습니다.

| Node | 역할 |
|------|------|
| `optimist` | 낙관론자 — 주제에 찬성·긍정 |
| `skeptic` | 회의론자 — 반박·비판 |

**중단 조건** (`debate_route`):
* `turn_count >= max_turns` → `END`
* 마지막 메시지에 `'패배 인정'` 포함 → `END`
* 그 외 → `optimist`로 돌아가 순환

In [4]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
print('WORKDIR :', WORKDIR)

WORKDIR : c:\Users\Admin\OneDrive\바탕 화면\실습\16일차_실습


In [5]:
from typing import Literal, Annotated
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    max_turns: int = 3
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'


llm = ChatOpenAI(model='gpt-5-nano', temperature=0.8, api_key=OPENAI_API_KEY)

In [6]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '찬성' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(content=f'토론 주제는 {state.topic} 이제부터는 사람 없이 AI끼리 토론을 진행합니다.')) # 첫 대화이면 토론 주제를 주고
    else:
        prompts.extend(state.chat_history) # 이전 대화가 있으면 이어서 대화

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문

    response = llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
             "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '반대' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history,
    ]
    response = llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

### `route` 함수와 Conditional Edge 구현
* `debate_route`는 **다음에 갈 Node 이름** 또는 `END`를 반환합니다.
* `add_conditional_edges('skeptic', debate_route)`

In [7]:
from langgraph.graph import StateGraph, START, END

def debate_route(state: DebateState):
    if state.turn_count >= state.max_turns:
        return END
    last_text = state.chat_history[-1].content if state.chat_history else ''
    if '패배 인정' in last_text:
        return END
    return 'optimist'


debate_workflow = StateGraph(DebateState)
debate_workflow.add_node('optimist', optimist_node)
debate_workflow.add_node('skeptic', skeptic_node)

debate_workflow.add_edge(START, 'optimist')
debate_workflow.add_edge('optimist', 'skeptic')
debate_workflow.add_conditional_edges('skeptic', debate_route)

debate_app = debate_workflow.compile()

In [8]:
init_debate = DebateState(topic='AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.')

for chunk in debate_app.stream(init_debate):
    print(chunk)

{'optimist': {'chat_history': [AIMessage(content='찬성 입장 요지:\n- 의료 분야에서 AI는 조기 진단과 맞춤 치료를 가능하게 해 신체적 고통을 줄이고 삶의 질을 높인다.\n- 교육은 AI가 학습 속도와 선호에 맞춘 개인화된 경로를 제공하여 모든 사람이 잠재력을 최대한 발휘하게 한다.\n- 반복적이거나 위험한 업무를 AI가 대체함으로써 사람들은 더 의미 있고 창의적인 활동에 집중할 수 있다.\n- 정서적 지지와 사회적 연결감을 확장하는 도구로 AI가 고립감을 줄이고 정신적 안녕을 향상시킨다.\n- 자원 관리와 재난 대응의 최적화를 통해 환경 지속 가능성과 안전망이 강화되어 전반적 행복을 증진한다.\n\n상대 의견에 대한 두 문장 이내 반박:\n상대의 위험 주장은 과장된 부분이 있지만, 위험은 설계와 정책으로 충분히 관리 가능하다. 재교육과 사회적 안전망, 그리고 투명한 거버넌스로 AI의 긍정적 효과를 극대화할 수 있다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2812, 'prompt_tokens': 117, 'total_tokens': 2929, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2560, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E14NfklT13epIQqhFNV

## 사회자 추가하기

토론 그래프에 **`moderator` Node**를 추가해 보세요.

* 매 라운드 끝에 양쪽 주장을 한 줄로 요약
* `debate_route`에서 `moderator`를 거친 뒤 `optimist`로 보내기
* 사회자가 '종료'를 언급해야 토론이 종료되도록 종료 조건 수정

In [26]:
class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'

In [20]:
class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    max_turns: int = 9
    last_speaker: Literal['optimist', 'skeptic', 'moderator'] = 'skeptic'

In [21]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '찬성' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
        )),
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(content=f'토론 주제는 {state.topic}'))
    else:
        prompts.extend(state.chat_history)

    response = llm.invoke(prompts)
    response.name = 'optimist'
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    }

def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '반대' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history,
    ]
    response = llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

In [17]:
def moderator_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 AI 토론 대회의 사회자입니다."
            "'찬성', '반대'측 AI 토론자 사이에서 토론을 진행하세요."
            "방금 끝난 라운드의 찬성·반대 주장을 각각 한 줄로 요약하세요."
            f"현재 {state.turn_count}라운드 / 최대 {state.max_turns}라운드입니다."
            "토론을 종료하려면 반드시 '종료'를 언급하세요."
            "토론이 너무 길어지거나 최대 라운드에 도달하면 강제로 '종료'를 외쳐 토론을 마무리하세요."
        ))
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(
            content=f'토론 주제는 {state.topic} 이제부터는 사람 없이 AI끼리 토론을 진행합니다.'
        ))
    else:
        prompts.extend(state.chat_history)

    response = llm.invoke(prompts)
    response.name = 'moderator'
    return {
        'chat_history': [response],
        'last_speaker': 'moderator',
    }


def debate_route(state: DebateState):
    last_text = state.chat_history[-1].content if state.chat_history else ''
    if '종료' in last_text:
        return END
    return 'optimist'

In [23]:
def moderator_node(state: DebateState) -> dict:
    if state.turn_count == 0:
        # 첫 턴: 오프닝
        system = (
            "당신은 AI 토론 대회의 사회자입니다."
            f"토론 주제는 '{state.topic}' 입니다."
            "주제를 소개하고 토론 규칙을 간단히 안내한 뒤 토론 시작을 선언하세요."
            "첫 오프닝에서는 절대 '종료'라고 말하지 마세요."
        )
        prompts = [SystemMessage(content=system)]
        if not state.chat_history:
            prompts.append(HumanMessage(
                content='이제 토론을 시작합니다. 사회자 오프닝을 해주세요.'
            ))
        else:
            prompts.extend(state.chat_history)
    else:
        # 이후 턴: 라운드 요약
        system = (
            "당신은 AI 토론 대회의 사회자입니다."
            "방금 끝난 라운드의 찬성·반대 주장을 각각 한 줄로 요약하세요."
            f"현재 {state.turn_count}라운드 / 최대 {state.max_turns}라운드입니다."
            f"{state.turn_count} >= {state.max_turns} 이면 반드시 '종료'를 외치세요."
            "그 외에는 '종료'를 말하지 마세요."
        )
        prompts = [SystemMessage(content=system), *state.chat_history]

    response = llm.invoke(prompts)
    response.name = 'moderator'
    return {
        'chat_history': [response],
        'last_speaker': 'moderator',
    }

def debate_route(state: DebateState):
    last_text = state.chat_history[-1].content if state.chat_history else ''
    if '종료' in last_text:
        return END
    return 'optimist'

In [24]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(DebateState)

workflow.add_node('optimist', optimist_node)
workflow.add_node('skeptic', skeptic_node)
workflow.add_node('moderator', moderator_node)

workflow.add_edge(START, 'moderator')
workflow.add_edge('optimist', 'skeptic')
workflow.add_edge('skeptic', 'moderator')          # skeptic → moderator (고정)
workflow.add_conditional_edges('moderator', debate_route)  # moderator → optimist | END

debate_app = workflow.compile()

In [25]:
init_debate = DebateState(topic='AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.')

for chunk in debate_app.stream(init_debate):
    print(chunk)

{'moderator': {'chat_history': [AIMessage(content='안녕하세요 여러분. 오늘의 토론에 오신 것을 환영합니다.\n\n주제\nAI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.\n\n토론 규칙(간단 안내)\n- 팀 구성: 두 팀으로 진행합니다. 찬성 측과 반대 측. 각 팀은 공개 발언자 1명씩을 기본으로 하되 필요 시 보조 발언자를 둘 수 있습니다.\n- 발언 순서: 찬성 측 → 반대 측 → 찬성 측 → 반대 측로 진행합니다. 각 차수마다 진행자가 시간 제약 내에서 진행합니다.\n- 발표 시간: 각 발언은 2분, 중간 반론/추가 주장도 각 팀 2분씩 허용합니다. 마무리 발언은 각 팀 1분으로 마무리합니다.\n- 질의응답: 필요 시 청중이나 상대 팀으로부터의 질의가 있을 수 있으며, 각 질의에 대해 발언자는 1회 답변으로 응대합니다.\n- 시간 관리: 제가 시간표를 알려드리고, 남은 시간도 시각적으로 안내드리겠습니다. 시간 엄수는 토론의 공정함을 위해 중요합니다.\n- 예의와 근거: 서로의 주장을 존중하고, 주장 뒷받침 근거와 출처를 명시합니다. 논리적이고 구체적인 증거를 우선합니다.\n- 규칙 준수: 발언 중 끼어들기 금지, 자료 인용 시 출처를 밝히기, 비방이나 인신공격 금지 등의 기본 예의를 지킵니다.\n\n참여자에게 당부\n- 각 팀은 가능한 구체적인 사례와 데이터를 인용해 주시고, 모순되는 주장에는 근거 제시로 대응해 주세요.\n- 진행자는 중립적 입장에서 시간과 흐름을 관리하고, 필요 시 요약이나 주제 리마인드를 제공합니다.\n\n자, 이제 토론을 시작하겠습니다. 찬성 측의 첫 발언자가 준비되면 시작해 주세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2244, 'prompt_tokens': 101, 'total_tokens': 2345, 'completion_tokens_details

KeyboardInterrupt: 